# 1. PROBLEM STATEMENT

## E-commerce platforms often struggle to convert users due to lack of personalization. Generic product recommendations fail to capture user intent, leading to lower engagement and missed revenue opportunities.

# 2. OBJECTIVE

## The goal of this project is to build a personalized recommendation system that suggests relevant products to users based on their past behavior, thereby improving conversion rates and customer experience.

# 3. DATA UNDERSTANDING

### The dataset contains customer transactions from an e-commerce platform.
Key variables used:

Customer ID (user identity)
Product ID (purchase behavior)

To reduce sparsity and improve computational efficiency, the analysis was restricted to the top 100 most frequently purchased products.

In [9]:
import pandas as pd
import sqlite3

# Connect to the SQLite database file
conn = sqlite3.connect("/Users/anayavaz/Downloads/olist.sqlite")

# SQL query to get all table names from the database
query = "SELECT name FROM sqlite_master WHERE type='table';"

# Load the result into a pandas DataFrame
tables_df = pd.read_sql_query(query, conn)

# Print all table names
print("Tables in the database:")
for table_name in tables_df["name"]:
    print(table_name)

# Close the database connection
conn.close()


Tables in the database:
product_category_name_translation
sellers
customers
geolocation
order_items
order_payments
order_reviews
orders
products
leads_qualified
leads_closed


In [10]:
import pandas as pd
import sqlite3

# Connect to your SQLite database
conn = sqlite3.connect("/Users/anayavaz/olist.sqlite")

# Load the tables into pandas DataFrames
orders = pd.read_sql_query("SELECT * FROM orders", conn)
order_items = pd.read_sql_query("SELECT * FROM order_items", conn)
customers = pd.read_sql_query("SELECT * FROM customers", conn)

# Show the first 5 rows of each table
print("Orders:")
display(orders.head())

print("Order Items:")
display(order_items.head())

print("Customers:")
display(customers.head())

# Close the connection
conn.close()


Orders:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


Order Items:


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


Customers:


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [11]:
# Step 1: Merge orders with customers using customer_id
orders_customers = pd.merge(
    orders,
    customers,
    on="customer_id",
    how="inner"
)

# Step 2: Merge the above result with order_items using order_id
df = pd.merge(
    orders_customers,
    order_items,
    on="order_id",
    how="inner"
)

# Step 3: Keep only the required columns
df = df[["customer_unique_id", "product_id"]]

# Step 4: Drop duplicate customer-product combinations
df = df.drop_duplicates()

# Step 5: Show the first 5 rows
df.head()


,customer_unique_id,product_id
0,7c396fd4830fd04220f754e42b4e5bff,87285b34884572647811a353c7ac498a
1,af07308b275d755c9edb36a90c618231,595fac2a385ac33a80bd5114aec74eb8
2,3a653a41f6f9fc3d2a113cf8398680e8,aa4383b373c6aca5d8797843e5594415
3,7c142cf63193a1473d2e66489a9ae977,d0b61bfb1de832b15ba9d266ca96e5b0
4,72632f0f9dd73dfee390c9b22eb56dd6,65266b2da20d04dbe00c5c2d3bb7859e


# 4. METHODOLOGY

## 🔹 Baseline Model (Popularity-Based)

### A baseline recommendation model was created using the most frequently purchased products. While simple, this approach lacks personalization and serves as a benchmark.

In [12]:
# Count how many times each product_id appears
product_counts = df["product_id"].value_counts()

# Sort is already done by value_counts() in descending order,
# then select the top 10 most popular products
top_products = product_counts.head(10)

# Print the result
print("Top 10 most popular products:")
print(top_products)


Top 10 most popular products:
product_id
99a4788cb24856965c36a24e339b6058    466
aca2eb7d00ea1a7b8ebd4e68314663af    430
422879e10f46682990de24d770e7f83d    349
d1c427060a0f73f6b889a5c7c61f2ac4    322
389d119b48cf3043d311335e499d9c6b    310
53b36df67ebb7c41585e8d54d6772e08    303
368c6c730842d78016ad823897a372db    289
53759a2ecddad2bb87a079a1f1519f73    283
154e7e31ebfa092203795c972e5804a6    268
2b4609f8948be18874494203496bc318    257
Name: count, dtype: int64


In [13]:
# Create a simple function to recommend the most popular products
def recommend_popular():
    return top_products.head(5)

# Print the recommendations
print("Top 5 recommended popular products:")
print(recommend_popular())


Top 5 recommended popular products:
product_id
99a4788cb24856965c36a24e339b6058    466
aca2eb7d00ea1a7b8ebd4e68314663af    430
422879e10f46682990de24d770e7f83d    349
d1c427060a0f73f6b889a5c7c61f2ac4    322
389d119b48cf3043d311335e499d9c6b    310
Name: count, dtype: int64


## 🔹 Improved Model (Filtered Popularity)

### To improve relevance, already purchased products were excluded from recommendations, ensuring users receive only new suggestions.

In [14]:
# Create a function to recommend products for a given user
def recommend_for_user(user_id):
    # Get all products already purchased by the user
    purchased_products = df[df["customer_unique_id"] == user_id]["product_id"]

    # Remove already purchased products from the popular products list
    remaining_products = top_products[~top_products.index.isin(purchased_products)]

    # Return the top 5 remaining products
    return remaining_products.head(5)

# Example: replace with a real customer_unique_id from your data
# print(recommend_for_user("your_user_id_here"))


In [15]:
recommend_for_user("7c396fd4830fd04220f754e42b4e5bff")

product_id
99a4788cb24856965c36a24e339b6058    466
aca2eb7d00ea1a7b8ebd4e68314663af    430
422879e10f46682990de24d770e7f83d    349
d1c427060a0f73f6b889a5c7c61f2ac4    322
389d119b48cf3043d311335e499d9c6b    310
Name: count, dtype: int64

## 🔹 Core Model (Collaborative Filtering)

### A user-based collaborative filtering model was implemented using cosine similarity. Users with similar purchase behavior were identified, and recommendations were generated based on products purchased by similar users.

In [8]:
# Memory-friendly version:
# Keep only the top 100 most popular products before creating the matrix

TOP_N = 100

# Find the top 100 most popular products
top_product_ids = df["product_id"].value_counts().head(TOP_N).index

# Keep only rows for those products
df_small = df[df["product_id"].isin(top_product_ids)]

# Create the user-item matrix
# Rows = customers
# Columns = top products
# Values = number of purchases/interactions
user_item_matrix = pd.crosstab(
    df_small["customer_unique_id"],
    df_small["product_id"]
)

# Optional: use a smaller integer type to save memory
user_item_matrix = user_item_matrix.astype("int16")

# Print shape
print("User-item matrix shape:", user_item_matrix.shape)

# Show first 5 rows
user_item_matrix.head()


User-item matrix shape: (12627, 100)


product_id,06edb72f1e0c64b14c5b79353f7abea3,08574b074924071f4e201e151b152b4e,0a57f7d2c983bcf8188589a5fea4a8da,0aabfb375647d9738ad0f7b4ea3653b1,0bcc3eeca39e1064258aa1e932269894,154e7e31ebfa092203795c972e5804a6,1613b819ab5dae53aead2dbb4ebdb378,165f86fe8b799a708a20ee4ba125c289,18b0e642cbae7251e60a64aa07dd9eb9,19c91ef95d509ea33eda93495c4d3481,...,dbb67791e405873b259e4656bf971246,e0cf79767c5b016251fe139915c59a26,e0d64dcfaa3b6db5c54ca298ae101d05,e44f675b60b3a3a2453ec36421e06f0f,e53e557d5a159f5aa2c5e995dfdf244b,ec2d43cc59763ec91694573b31f1c29a,ee0c1cf2fbeae95205b4aa506f1469f0,ee406bf28024d97771c4b1e8b7e8e219,f1c7f353075ce59d8a6f3cf58f419c9c,fb55982be901439613a95940feefd9ee
customer_unique_id,,,,,,,,,,,,,,,,,,,,,
00196c4c9a3af7dd2ad10eade69c926f,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
001ae5a1788703d64536c30362503e49,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
002ae492472e45ad6ebeb7a625409392,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
002bdeb33da5b1b3ce8b9c822f749c82,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
002ed12115742033f015cb3c269ccf68,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [16]:
TOP_N = 100

# Find the top 100 most popular products
top_product_ids = df["product_id"].value_counts().head(TOP_N).index

# Keep only rows for those products
df_small = df[df["product_id"].isin(top_product_ids)]

# Create the user-item matrix
user_item_matrix = pd.crosstab(
    df_small["customer_unique_id"],
    df_small["product_id"]
)

# Save memory
user_item_matrix = user_item_matrix.astype("int16")

# Print useful outputs
print("df shape:", df.shape)
print("df_small shape:", df_small.shape)
print("User-item matrix shape:", user_item_matrix.shape)

user_item_matrix.head()


df shape: (101987, 2)
df_small shape: (12883, 2)
User-item matrix shape: (12627, 100)


product_id,06edb72f1e0c64b14c5b79353f7abea3,08574b074924071f4e201e151b152b4e,0a57f7d2c983bcf8188589a5fea4a8da,0aabfb375647d9738ad0f7b4ea3653b1,0bcc3eeca39e1064258aa1e932269894,154e7e31ebfa092203795c972e5804a6,1613b819ab5dae53aead2dbb4ebdb378,165f86fe8b799a708a20ee4ba125c289,18b0e642cbae7251e60a64aa07dd9eb9,19c91ef95d509ea33eda93495c4d3481,...,dbb67791e405873b259e4656bf971246,e0cf79767c5b016251fe139915c59a26,e0d64dcfaa3b6db5c54ca298ae101d05,e44f675b60b3a3a2453ec36421e06f0f,e53e557d5a159f5aa2c5e995dfdf244b,ec2d43cc59763ec91694573b31f1c29a,ee0c1cf2fbeae95205b4aa506f1469f0,ee406bf28024d97771c4b1e8b7e8e219,f1c7f353075ce59d8a6f3cf58f419c9c,fb55982be901439613a95940feefd9ee
customer_unique_id,,,,,,,,,,,,,,,,,,,,,
00196c4c9a3af7dd2ad10eade69c926f,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
001ae5a1788703d64536c30362503e49,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
002ae492472e45ad6ebeb7a625409392,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
002bdeb33da5b1b3ce8b9c822f749c82,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
002ed12115742033f015cb3c269ccf68,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [18]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Calculate cosine similarity between all users
# This creates a user-user similarity matrix
similarity_matrix = cosine_similarity(user_item_matrix)

# Convert the result into a pandas DataFrame
# Use the same user IDs for both rows and columns
user_similarity = pd.DataFrame(
    similarity_matrix,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

# Show the first 5 rows
user_similarity.head()


customer_unique_id,00196c4c9a3af7dd2ad10eade69c926f,001ae5a1788703d64536c30362503e49,002ae492472e45ad6ebeb7a625409392,002bdeb33da5b1b3ce8b9c822f749c82,002ed12115742033f015cb3c269ccf68,002ef00822613c94613e60e03b169fef,002feefec5af0a3b26ee7839c66d205e,00344274804f3b8003de1b0562ae01df,0036a074f98b80c4f1fc33dbbcf9c552,004b45ec5c64187465168251cd1c9c2f,...,ffc9d8a98ebd09a78c8adb729c3cfa2a,ffcf1dc25f2222aea1aef48841f20f3a,ffdcd85e0e81fc93f49bef4d54a4e61f,ffe39c116d96cd81dcdfd9dcca655cd1,ffe6305176b9431a3eda3cf8904d7eb7,ffe9be10b9a58c5464d833e8b1b2c632,ffe9e41fbd14db4a7361347c56af5447,ffee94d548cef05b146d825a7648dab4,ffeefd086fc667aaf6595c8fe3d22d54,ffff5962728ec6157033ef9805bacc48
customer_unique_id,,,,,,,,,,,,,,,,,,,,,
00196c4c9a3af7dd2ad10eade69c926f,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
001ae5a1788703d64536c30362503e49,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
002ae492472e45ad6ebeb7a625409392,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
002bdeb33da5b1b3ce8b9c822f749c82,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
002ed12115742033f015cb3c269ccf68,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [24]:
# Collaborative filtering recommendation function
def recommend_cf(user_id):
    # Check if the user exists in the user-item matrix
    if user_id not in user_item_matrix.index:
        return "User not found"

    # Get similarity scores for the given user
    similar_users = user_similarity.loc[user_id]

    # Sort similar users from highest to lowest similarity
    # Remove the user itself
    similar_users = similar_users.sort_values(ascending=False).drop(user_id)

    # Select the top 5 similar users
    top_5_similar_users = similar_users.head(5).index

    # Get products already bought by the current user from the full dataframe
    user_products = set(
        df.loc[df["customer_unique_id"] == user_id, "product_id"]
    )

    # Get products bought by the top 5 similar users from the full dataframe
    similar_users_products = df.loc[
        df["customer_unique_id"].isin(top_5_similar_users),
        "product_id"
    ]

    # Remove products the current user has already bought
    candidate_products = similar_users_products[
        ~similar_users_products.isin(user_products)
    ]

    # Get the top 5 most frequent remaining products
    recommendations = candidate_products.value_counts().head(5).index.tolist()

    # Fallback: if no recommendations are found, recommend popular unseen products
    if len(recommendations) < 5:
        popular_unseen = (
            df.loc[~df["product_id"].isin(user_products), "product_id"]
            .value_counts()
            .index
            .tolist()
        )

        for product_id in popular_unseen:
            if product_id not in recommendations:
                recommendations.append(product_id)
            if len(recommendations) == 5:
                break

    return recommendations

# Example test
test_user = user_item_matrix.index[0]
print(recommend_cf(test_user))


['e8f732eea94996831b9af41525a9ce3d', '99a4788cb24856965c36a24e339b6058', 'aca2eb7d00ea1a7b8ebd4e68314663af', '422879e10f46682990de24d770e7f83d', 'd1c427060a0f73f6b889a5c7c61f2ac4']


In [25]:
test_user = user_item_matrix.index[0]

already_bought = set(df[df["customer_unique_id"] == test_user]["product_id"])
recommended = recommend_cf(test_user)

print("Recommended products:", recommended)
print("Already bought by user:", already_bought)

print("Any overlap?", any(product in already_bought for product in recommended))


Recommended products: ['e8f732eea94996831b9af41525a9ce3d', '99a4788cb24856965c36a24e339b6058', 'aca2eb7d00ea1a7b8ebd4e68314663af', '422879e10f46682990de24d770e7f83d', 'd1c427060a0f73f6b889a5c7c61f2ac4']
Already bought by user: {'18b0e642cbae7251e60a64aa07dd9eb9'}
Any overlap? False


# 5. KEY INSIGHTS

## 💡 Insight 1: Sales Concentration

### A small subset of products drives a large portion of total purchases, indicating a strong popularity bias in customer behavior.

## 💡 Insight 2: User Behavior Clustering

### Customers exhibit similar purchasing patterns, allowing segmentation based on behavioral similarity.

## 💡 Insight 3: Personalization vs Generic

### Collaborative filtering provides more relevant recommendations compared to popularity-based methods, improving the likelihood of conversion.

## 💡 Insight 4: Cross-Selling Opportunity

### Users with similar behavior often purchase related products, highlighting opportunities for cross-selling and bundling strategies.

## 💡 Insight 5: Trade-off 

### Restricting the model to the top 100 products improves computational efficiency but may reduce recommendation diversity by excluding niche products.

# 6. BUSINESS IMPACT

## If implemented in a real-world scenario, this system could:

### - Increase conversion rates through personalized recommendations
### - Improve customer retention via better user experience
### - Boost average order value through cross-selling
### - Enable targeted marketing campaigns based on user similarity

# 7. FUTURE IMPROVEMENTS

## Future enhancements could include:

### - Incorporating product metadata (category, price)
### - Using advanced models (matrix factorization)
### - Integrating real-time user behavior (clickstream data)
### - Expanding beyond top 100 products for better diversity